# Bronze Layer: Ingestion & Extraction Notebook
Extracts source data from the local database via the Bore TCP tunnel into Databricks using PySpark JDBC, secure secrets, and centralized task logging.

In [ ]:
# 1. Link Centralized Logger Utility
# Executes the logger notebook to make get_task_logger available
%run ../../src/utilities/logger

In [ ]:
# 2. Initialize SparkSession and Task Logger
from pyspark.sql import SparkSession

# Ensure SparkSession is active
spark = SparkSession.builder.getOrCreate()

# Initialize logger for this notebook execution
logger = get_task_logger(notebook_name_override="bronze_extraction")
logger.info("=== Starting Bronze Extraction Notebook Execution ===")

In [ ]:
# 3. Interactive Widget for Dynamic Tunnel Port
dbutils.widgets.text("tunnel_port", "46985", "Bore Tunnel Port")
db_port = dbutils.widgets.get("tunnel_port")

logger.info(f"Using Bore Tunnel Port: {db_port}")
print(f"Active Tunnel Port: {db_port}")

In [ ]:
# 4. Retrieve Credentials Securely from Databricks Secrets
SECRET_SCOPE = "wanderbricks_scope"

db_user = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_user")
db_password = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_password")
db_host = dbutils.secrets.get(scope=SECRET_SCOPE, key="tunnel_host")
db_name = dbutils.secrets.get(scope=SECRET_SCOPE, key="mysql_db")

logger.info(f"Retrieved credentials from scope '{SECRET_SCOPE}' for user '{db_user}'")
print(f"Connecting to Database: {db_name} at Host: {db_host} as User: {db_user}")

In [ ]:
# 5. Assemble JDBC URL and Test Extraction
jdbc_url = f"jdbc:mysql://{db_host}:{db_port}/{db_name}?useSSL=false&allowPublicKeyRetrieval=true"

connection_properties = {
    "user": db_user,
    "password": db_password,
    "driver": "com.mysql.cj.jdbc.Driver"
}

table_to_extract = "countries"
logger.info(f"Extracting table '{table_to_extract}' via JDBC URL: {jdbc_url}")

# Read table into PySpark DataFrame
df = spark.read.jdbc(
    url=jdbc_url,
    table=table_to_extract,
    properties=connection_properties
)

record_count = df.count()
logger.info(f"Successfully extracted {record_count} rows from table '{table_to_extract}'")
print(f"Extracted {record_count} rows from {table_to_extract}:")

display(df)